# Simple CNN Classifier

Train a small 1D CNN on preprocessed_dataset.pt to predict keyboard characters (classification).
Standalone notebook — no GIK trainer. Uses Focal Loss.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import numpy as np

PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pretraining import load_preprocessed_dataset, get_class_weights
from src.Constants.char_to_key import CHAR_TO_INDEX, NUM_CLASSES, INDEX_TO_CHAR
from ml.models.loss_functions.custom_losses import FocalLoss
from src.visualisation.visualisation import compute_confusion_matrix_40x40, plot_virtual_keyboard_heatmap

torch.manual_seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Device: {DEVICE}")

## Config & Load Data

In [ ]:
DATA_PATH = "data_hazel_7/processed_dataset.pt"
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-3
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15

In [ ]:
dataset = load_preprocessed_dataset(
    DATA_PATH,
    char_to_index=CHAR_TO_INDEX,
    is_one_hot_labels=False,
    return_class_id=False,
    add_prev_char=True,
)
print(f"Dataset: {len(dataset)} samples, input_dim={dataset.input_dim}, num_classes={NUM_CLASSES}")

In [ ]:
n = len(dataset)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)
n_test = n - n_train - n_val

train_ds = Subset(dataset, range(0, n_train))
val_ds = Subset(dataset, range(n_train, n_train + n_val))
test_ds = Subset(dataset, range(n_train + n_val, n))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## Simple CNN Model

In [ ]:
class SimpleCNNClassifier(nn.Module):
    """Small 1D CNN for character classification."""
    def __init__(self, input_dim: int, num_classes: int = 40, hidden_dim: int = 64, kernel_sizes=(3, 5, 7), dropout: float = 0.2):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
        )
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(hidden_dim, hidden_dim, k, padding=k//2),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
            for k in kernel_sizes
        ])
        self.head = nn.Linear(hidden_dim * len(kernel_sizes), num_classes)

    def forward(self, x):
        # x: (B, T, D)
        x = self.proj(x)
        x = x.transpose(1, 2)  # (B, H, T)
        conv_outputs = []
        for conv in self.convs:
            c = conv(x)
            c = F.adaptive_max_pool1d(c, 1).squeeze(-1)
            conv_outputs.append(c)
        x = torch.cat(conv_outputs, dim=-1)
        return self.head(x)

## Train

In [ ]:
class_weights = get_class_weights(DATA_PATH, train_ratio=0.8, split_strategy="contiguous").to(DEVICE)
criterion = FocalLoss(gamma=2.0, alpha=class_weights)
model = SimpleCNNClassifier(input_dim=dataset.input_dim, num_classes=NUM_CLASSES, hidden_dim=64, kernel_sizes=(3, 5, 7), dropout=0.2).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

## Model Architecture

In [ ]:
print(model)
print("\n" + "=" * 60)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        x, y = [b.to(device) for b in batch]
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            x, y = [b.to(device) for b in batch]
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += x.size(0)
    return total_loss / total if total else 0.0, correct / total if total else 0.0

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val acc: {val_acc:.4f}")

## Results

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)

print("=" * 60)
print("Simple CNN Classifier Results (Focal Loss)")
print("=" * 60)
print(f"Focal Loss (gamma=2.0):")
print(f"  Val loss:  {history['val_loss'][-1]:.4f}")
print(f"  Test loss: {test_loss:.4f}")
print(f"Character accuracy:")
print(f"  Val acc:  {history['val_acc'][-1]:.4f} ({history['val_acc'][-1]*100:.2f}%)")
print(f"  Test acc: {test_acc:.4f} ({test_acc*100:.2f}%)")
print("=" * 60)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].set_title("Focal Loss")
axes[1].plot(history["val_acc"], label="Val acc", color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].set_title("Validation Accuracy")
plt.tight_layout()
plt.show()

## Keyboard Heatmap

In [ ]:
cm_test = compute_confusion_matrix_40x40(test_ds, model, DEVICE, coord_dict=None)

In [ ]:
for anchor in ['e', 'a', 's', 'd', ' ']:
    plot_virtual_keyboard_heatmap(cm_test, anchor, 'Test')